# 03 — Полный бенчмарк на всём датасете

Прогон baseline (whisper) + alignment (per-lang XLSR-53 / MMS) на всех 9 треках Yandex-датасета. Лирики — LRCLib API.

Один прогон → CSV-таблица docs/results.csv. Цифры подбиваются в docs/bench_log.md локально потом.

Запускать на Colab T4. Время прогона: ~15-20 минут (whisper большой, грузится один раз).

In [ ]:
import torch
assert torch.cuda.is_available()
print(torch.cuda.get_device_name(0))

In [ ]:
!rm -rf unbake-test
!git clone -q https://github.com/RezPoint/unbake-test.git
%cd unbake-test
!pip install -q faster-whisper transformers torchaudio librosa jiwer pydantic requests

In [ ]:
# Скачиваем датасет + лирики (LRCLib с fallback ASCII-fold + no-artist)
!python notebooks/download_dataset.py
!python -m src.lyrics

In [ ]:
# Сбрасываем кэш транскриптов: replays нужен чтобы real RTF замерился
# (после фикса torch.cuda.synchronize в src/align.py)
import shutil
shutil.rmtree("data/transcripts", ignore_errors=True)
print("cache cleared")

In [ ]:
# Полный батч-прогон (baseline + alignment per track)
!python -m src.eval.batch_runner

In [ ]:
import pandas as pd
df = pd.read_csv("docs/results.csv")
print(df.to_string(index=False))
df.describe()

In [ ]:
!cd data/transcripts && tar czf /content/transcripts.tar.gz *.json
!cp docs/results.csv /content/results.csv
print("downloaded:")
print("  /content/transcripts.tar.gz")
print("  /content/results.csv")